In [2]:

import os
from pathlib import Path
import duckdb

#move to the project root if needed
if Path.cwd().name == "notebooks":
    os.chdir("..")


con = duckdb.connect("data/olist.duckdb")

con.execute(Path("sql/01_load_raw.sql").read_text())

con.sql("SHOW TABLES").df()

c:\Users\Bugra\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,name
0,raw_category_translation
1,raw_customers
2,raw_order_items
3,raw_orders
4,raw_payments
5,raw_products
6,raw_reviews
7,raw_sellers


In [3]:
con.sql("SELECT COUNT(*) AS total_orders FROM raw_orders").df

<bound method DuckDBPyRelation.df of ┌──────────────┐
│ total_orders │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘
>

In [4]:
con.sql("DESCRIBE raw_orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


In [5]:
con.sql("""
SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM raw_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM raw_order_items
UNION ALL SELECT 'customers', COUNT(*) FROM raw_customers
UNION ALL SELECT 'payments', COUNT(*) FROM raw_payments
UNION ALL SELECT 'reviews', COUNT(*) FROM raw_reviews
UNION ALL SELECT 'products', COUNT(*) FROM raw_products
UNION ALL SELECT 'sellers', COUNT(*) FROM raw_sellers;
""").df()

,table_name,n_rows
0,orders,99441
1,order_items,112650
2,customers,99441
3,payments,103886
4,reviews,99224
5,products,32951
6,sellers,3095


In [6]:
#checked for duplicate orders

con.sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT order_id) AS n_unique_orders
FROM raw_orders; 
""").df()

,n_rows,n_unique_orders
0,99441,99441


In [7]:
#checked for orders with multiple reviews

con.sql("""
SELECT COUNT(*) AS orders_with_several_reviews
FROM (
    SELECT order_id
    FROM raw_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1);
""").df()

,orders_with_several_reviews
0,547


In [8]:
#keep the latest review for each order, with this code, we check the number for each delivery status

con.sql("""
SELECT order_status,
       COUNT (*) AS n_orders
FROM raw_orders
GROUP BY  order_status
ORDER BY n_orders DESC;
""").df()

,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [9]:
con.sql("""
SELECT COUNT(*) AS non_delivered_orders
FROM raw_orders
WHERE order_status <> 'delivered';
""").df()

,non_delivered_orders
0,2963


In [10]:
con.sql("""
SELECT COUNT (*) AS delivered_without_date
FROM raw_orders
WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NULL;
""").df()

,delivered_without_date
0,8


In [11]:
#Look at the orders by date. We determined that we should include from January 2017 through August 2018

con.sql("""
SELECT date_trunc('month', order_purchase_timestamp) AS month,
       COUNT(*) AS n_orders
FROM raw_orders
GROUP BY month
ORDER BY month;
""").df()

,month,n_orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [12]:
#number of products without categories. We should replace them with unknown

con.sql("""
SELECT COUNT(*) AS products_without_categories
FROM raw_products
WHERE product_category_name IS NULL
""").df()

,products_without_categories
0,610


In [13]:
#we should use customer_unique_id when analyzing distinct customers

con.sql("""
SELECT COUNT(DISTINCT customer_id) AS customer_ids,
       COUNT(DISTINCT customer_unique_id) AS real_customers
FROM raw_customers;
""").df()

,customer_ids,real_customers
0,99441,96096


In [17]:
from pathlib import Path

con.execute(
    Path("sql/03_star_schema.sql").read_text(encoding="utf-8")
)

con.sql("SELECT * FROM fact_orders LIMIT 5"
        ).df()

,order_id,customer_key,order_date,order_status,order_value,delivery_days,days_late,is_late,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02,delivered,38.71,8,-8,0,4
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24,delivered,141.46,14,-6,0,4
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08,delivered,179.12,9,-18,0,5
3,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13,delivered,28.62,3,-10,0,5
4,a4591c265e18cb1dcee52889e2d8acc3,80bb27c7c16e8f973207a5086ab329e2,2017-07-09,delivered,175.26,17,-6,0,4


In [ ]:
# we check if any rows were duplicated or lost

con.sql("""
SELECT
    (SELECT COUNT(*) FROM raw_orders) AS raw_orders,
    (SELECT COUNT(*) FROM fact_orders) AS fact_orders,
    (SELECT COUNT(*) FROM raw_order_items) AS raw_items,
    (SELECT COUNT(*) FROM fact_order_items) AS fact_items;
""").df()

,raw_orders,fact_orders,raw_items,fact_items
0,99441,99441,112650,112650


In [36]:
con.sql("""
SELECT *
FROM dim_date LIMIT 5
""").df()

,date_key,year,month,year_month,quarter,weekday
0,2016-01-01,2016,1,2016-01,1,Friday
1,2016-01-02,2016,1,2016-01,1,Saturday
2,2016-01-03,2016,1,2016-01,1,Sunday
3,2016-01-04,2016,1,2016-01,1,Monday
4,2016-01-05,2016,1,2016-01,1,Tuesday


In [40]:
con.sql("""
SELECT 'customers' AS dimension,
        COUNT(*) AS rows,
        COUNT(DISTINCT customer_key) AS unique_keys
FROM dim_customer

UNION ALL

SELECT 'products', 
        COUNT(*),
        COUNT(DISTINCT product_key)
FROM dim_product

UNION ALL

SELECT 'seller',
       COUNT(*),
       COUNT(DISTINCT seller_key)
FROM dim_seller

UNION ALL

SELECT 'dates',
       COUNT(*),
       COUNT(DISTINCT date_key)
FROM dim_date;
""").df()

,dimension,rows,unique_keys
0,customers,96096,96096
1,products,32951,32951
2,seller,3095,3095
3,dates,1096,1096
